# Multi-Label Classification

## Imports

In [1]:
import ast
import glob
import hashlib
import inspect
import json
import os
import pickle
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from numpy.typing import ArrayLike
from sklearn import preprocessing
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    hamming_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.exceptions import NotFittedError
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from tqdm.auto import tqdm  # widgets when ipywidgets exists, text otherwise
from typing import Dict, Union

In [2]:
load_dotenv()

RANDOM_STATE = int(os.getenv("RANDOM_STATE", "42"))
TEST_SIZE = float(os.getenv("TEST_SIZE", "0.2"))

# Stage of this session: "embed" (cache embeddings), "train" (fit the
# candidates of MLC_UNITS from cached embeddings), "assemble" (select, evaluate
# on the test set, write models/mappings) or "all" (the three in one session).
STAGES = ("embed", "train", "assemble", "all")
STAGE = os.getenv("MLC_STAGE", "all").strip().lower() or "all"
if STAGE not in STAGES:
    raise ValueError(f"MLC_STAGE={STAGE!r}; expected one of {STAGES}")
print(f"MLC_STAGE: {STAGE}")

os.chdir(os.getenv("ROOT", "./"))
np.random.seed(seed=RANDOM_STATE)
random.seed(a=RANDOM_STATE)

In [3]:
import MLC
import notebooks.checkpoints as ck
import notebooks.utils as su

[nltk_data] Downloading package stopwords to /home/cela/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/cela/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Module and System classes

`predict` gives the hard labels; `predict_proba` gives per-label scores used only for AUC. A module's score for a label is the positive-class score (`predict_proba` column) of the model selected for that label. System scores follow the hierarchy: general labels take the general module's scores; relevance score = min(1 - score("other"), max score of the non-"other" general labels); a child label's score = min(gate, child score) with gate = min(relevance score, score of its parent general label). Thresholding these scores at 0.5 need not reproduce `predict`: CLR (vote fractions vs the virtual label), LP and PSt (most probable labelset vs label marginals), RAkEL and HOMER predict from their own rules, and the hard hierarchy gates children on the predicted parent.

In [ ]:
class ModuleSystem(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        models: Dict[str, Dict[str, Union[str, float, BaseEstimator]]],
        mlb: preprocessing.MultiLabelBinarizer,
        metadata=None,
    ):
        self.models = models
        self.mlb = mlb
        self.metadata = metadata  # run provenance: NON_PRODUCTION config, embedder inputs
        self.classes = mlb.classes_

    def embedder_name(self, module_name: str = "module") -> str:
        """
        Name of the embedder the per-label models were trained on, parsed from
        the model names "<METHOD>[<ALG>] - <embedder name>".
        """
        names = set()
        for idx, model_info in self.models.items():
            full_name = model_info.get("name") if isinstance(model_info, dict) else None
            if not full_name or " - " not in full_name:
                raise ValueError(
                    f"Module '{module_name}': model {idx} has no embedder in its "
                    f"name ({full_name!r}); expected '<METHOD>[<ALG>] - <embedder>'."
                )
            names.add(full_name.rsplit(" - ", 1)[1])
        if len(names) != 1:
            raise ValueError(
                f"Module '{module_name}': labels disagree on the embedder "
                f"({sorted(names)}); expected exactly one."
            )
        return names.pop()

    def _per_label(self, X: ArrayLike, method: str) -> ArrayLike:
        """Column idx of the output of the model selected for label idx; a
        model selected for several labels is evaluated once."""
        out = np.zeros((X.shape[0], len(self.models)))
        cache = {}
        for idx, model_info in self.models.items():
            name = model_info["name"]
            if name not in cache:
                cache[name] = np.asarray(getattr(model_info["model"], method)(X))
            out[:, int(idx)] = cache[name][:, int(idx)]
        return out

    def predict(self, X_test: ArrayLike) -> ArrayLike:
        """
        Predict labels for each component using the chosen model in the dictionary.

        Parameters
        ----------
        X_test : ArrayLike
            Test data.

        Returns
        -------
        Y_pred : ArrayLike
            Predicted labels for each component.
        """
        return self._per_label(X_test, "predict")

    def predict_proba(self, X_test: ArrayLike) -> ArrayLike:
        """Per-label positive scores: column idx of the ``predict_proba`` of
        the model selected for label idx (scores, not calibrated in general)."""
        return self._per_label(X_test, "predict_proba")


class System(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        gn: ModuleSystem,
        cs: ModuleSystem,
        os: ModuleSystem,
        sw: ModuleSystem,
        hw: ModuleSystem,
        metadata=None,
    ):
        self.gn = gn
        self.cs = cs
        self.os = os
        self.sw = sw
        self.hw = hw
        self.metadata = metadata  # run provenance (see ModuleSystem)

    def _modules(self):
        return {"gn": self.gn, "cs": self.cs, "os": self.os, "sw": self.sw, "hw": self.hw}

    def _layout(self):
        """(module key, parent general label, output column slice, child
        columns) of each child module; child columns skip its "other" class."""
        gn_dim = len(self.gn.classes)  # 6
        cs_dim = len(self.cs.classes) - 1  # 2
        os_dim = len(self.os.classes) - 1  # 4
        sw_dim = len(self.sw.classes) - 1  # 1
        hw_dim = len(self.hw.classes) - 1  # 3
        a = gn_dim
        b = a + cs_dim
        c = b + os_dim
        d = c + sw_dim
        e = d + hw_dim  # 16
        return e, [
            ("cs", "computer security", slice(a, b), list(range(cs_dim))),
            ("os", "operating systems", slice(b, c), [0, 1] + list(range(3, os_dim + 1))),
            ("sw", "software", slice(c, d), list(range(sw_dim))),
            ("hw", "hardware", slice(d, e), list(range(hw_dim))),
        ]

    def _inputs(self, texts: pd.Series, embeddings=None) -> dict:
        """Module key -> matrix of the embedder it was trained on.  With
        ``embeddings`` ({embedder name: matrix aligned with texts}) nothing is
        embedded here; a missing embedder then raises."""
        sources = {source["name"]: source for source in EMBEDDING_SOURCES}
        X_by_embedder = {}  # embedder name -> N x D matrix, computed once
        X = {}
        for key, module in self._modules().items():
            embedder = module.embedder_name(key)
            if embedder not in sources:
                raise ValueError(
                    f"Module '{key}': unknown embedder {embedder!r}; "
                    f"expected one of {sorted(sources)}."
                )
            if embedder not in X_by_embedder:
                if embeddings is not None:
                    if embedder not in embeddings:
                        raise ValueError(f"Module '{key}': no embeddings given for {embedder!r}.")
                    X_by_embedder[embedder] = np.asarray(embeddings[embedder])
                    if X_by_embedder[embedder].shape[0] != len(texts):
                        raise ValueError(f"Embeddings for {embedder!r} do not match the texts.")
                else:
                    source = sources[embedder]
                    X_by_embedder[embedder] = su.preprocess_texts(
                        list_str=texts,
                        model_path=source["model-path"],
                        embedding_dim=source["embedding-dim"],
                    )
            X[key] = X_by_embedder[embedder]
        return X

    def predict(self, texts: pd.Series, embeddings=None) -> ArrayLike:
        Y_cols, layout = self._layout()
        X = self._inputs(texts, embeddings)
        Y = np.zeros((texts.shape[0], Y_cols))  # N x 16
        gn_dim = len(self.gn.classes)
        other = list(self.gn.classes).index("other")  # 4

        y_gn = self.gn.predict(X["gn"])  # N x 6
        Y[:, :gn_dim] = y_gn
        no_label = y_gn.sum(axis=1) == 0
        is_other = y_gn[:, other] == 1
        Y[no_label, other] = 1.0
        Y[is_other, :] = 0.0
        Y[is_other, other] = 1.0
        active = ~no_label & ~is_other
        classes = list(self.gn.classes)
        modules = self._modules()
        for key, parent, cols, child_cols in layout:
            # Children only where the parent is predicted (and not "other");
            # rows are predicted independently, so a batch equals row-by-row.
            rows = active & (y_gn[:, classes.index(parent)] == 1)
            if rows.any():
                Y[rows, cols] = modules[key].predict(X[key][rows])[:, child_cols]
        return Y

    def relevance_score(self, texts: pd.Series, embeddings=None, _X=None) -> ArrayLike:
        """min(1 - score("other"), max score of the non-"other" general labels)."""
        X = self._inputs(texts, embeddings) if _X is None else _X
        s_gn = self.gn.predict_proba(X["gn"])
        other = list(self.gn.classes).index("other")
        rest = np.delete(s_gn, other, axis=1)
        return np.minimum(1.0 - s_gn[:, other], rest.max(axis=1))

    def predict_proba(self, texts: pd.Series, embeddings=None) -> ArrayLike:
        """Hierarchical scores in the column order of ``predict``: general =
        module scores; child = min(gate, child score), gate = min(relevance
        score, parent label score)."""
        Y_cols, layout = self._layout()
        X = self._inputs(texts, embeddings)
        S = np.zeros((texts.shape[0], Y_cols))
        gn_dim = len(self.gn.classes)
        s_gn = self.gn.predict_proba(X["gn"])
        S[:, :gn_dim] = s_gn
        relevance = self.relevance_score(texts, _X=X)
        classes = list(self.gn.classes)
        modules = self._modules()
        for key, parent, cols, child_cols in layout:
            gate = np.minimum(relevance, s_gn[:, classes.index(parent)])
            s_child = modules[key].predict_proba(X[key])[:, child_cols]
            S[:, cols] = np.minimum(gate[:, None], s_child)
        return S

    def predict_targets(self, texts: pd.Series, embeddings=None) -> pd.Series:
        Y = self.predict(texts, embeddings)  # N x 16
        gn_dim = len(self.gn.classes)  # 6
        cs_dim = len(self.cs.classes) - 1  # 2
        os_dim = len(self.os.classes) - 1  # 4
        sw_dim = len(self.sw.classes) - 1  # 1

        y_gn = Y[:, :gn_dim]
        y_cs = Y[:, gn_dim : gn_dim + cs_dim]
        y_os = Y[:, gn_dim + cs_dim : gn_dim + cs_dim + os_dim]
        y_sw = Y[:, gn_dim + cs_dim + os_dim : gn_dim + cs_dim + os_dim + sw_dim]
        y_hw = Y[:, gn_dim + cs_dim + os_dim + sw_dim :]

        gn_ctg = self.gn.mlb.inverse_transform(y_gn)
        cs_ctg = self.cs.mlb.inverse_transform(
            np.column_stack((y_cs, np.zeros((Y.shape[0], 1))))
        )
        os_ctg = self.os.mlb.inverse_transform(
            np.column_stack((y_os[:, :2], np.zeros((Y.shape[0], 1)), y_os[:, 2:]))
        )
        sw_ctg = self.sw.mlb.inverse_transform(
            np.column_stack((y_sw, np.zeros((Y.shape[0], 1))))
        )
        hw_ctg = self.hw.mlb.inverse_transform(
            np.column_stack((y_hw, np.zeros((Y.shape[0], 1))))
        )

        return pd.Series(
            data=[
                list(set(gni + csi + osi + swi + hwi))
                for gni, csi, osi, swi, hwi in zip(
                    gn_ctg, cs_ctg, os_ctg, sw_ctg, hw_ctg
                )
            ],
            index=texts.index,
        )

    def targets_matrix(self, targets: pd.Series) -> ArrayLike:
        """True label matrix in the column order of ``predict``."""
        gn_ctg = self.gn.mlb.transform(
            [
                [target for target in targetsi if target in self.gn.classes]
                for targetsi in targets
            ]
        )
        cs_ctg = self.cs.mlb.transform(
            [
                [target for target in targetsi if target in self.cs.classes]
                for targetsi in targets
            ]
        )
        os_ctg = self.os.mlb.transform(
            [
                [target for target in targetsi if target in self.os.classes]
                for targetsi in targets
            ]
        )
        sw_ctg = self.sw.mlb.transform(
            [
                [target for target in targetsi if target in self.sw.classes]
                for targetsi in targets
            ]
        )
        hw_ctg = self.hw.mlb.transform(
            [
                [target for target in targetsi if target in self.hw.classes]
                for targetsi in targets
            ]
        )
        return np.hstack(
            (
                gn_ctg,
                cs_ctg[:, :-1],
                os_ctg[:, :2],
                os_ctg[:, 3:],
                sw_ctg[:, :-1],
                hw_ctg[:, :-1],
            )
        )

    def evaluate(self, texts: pd.Series, targets: pd.Series, embeddings=None) -> dict[str, float]:
        """Hard predictions for the label metrics, hierarchical scores for AUC."""
        Y = self.targets_matrix(targets)
        Y_pred = self.predict(texts, embeddings)
        Y_score = self.predict_proba(texts, embeddings)

        return su.assess(Y, Y_pred, Y_score)

    def evaluate_filter(self, texts: pd.Series, targets: pd.Series, embeddings=None) -> dict[str, float]:
        Y_test = pd.Series([1 if t else 0 for t in targets])
        Y_pred = self.predict(texts, embeddings)

        return su.assess(Y_test, Y_pred[:, 4:5])

    def classes(self):
        return (
            list(self.gn.classes)
            + list(self.cs.classes[:-1])
            + list(self.os.classes[:2])
            + list(self.os.classes[3:])
            + list(self.sw.classes[:-1])
            + list(self.hw.classes[:-1])
        )

## Configuration

In [ ]:
# Declarative experiment configuration.  The source text of this cell is part
# of the checkpoint run identity (with the code and data hashes).
CONFIG_CELL_SOURCE = globals()["_ih"][-1]  # IPython input history: this cell

# method key -> (class, fixed params, grid {short name: (param, values)}).
# Every grid point is a separate named candidate selected on validation, e.g.
# "PST(p=25,b=1)[RF]".  CC keeps the natural label order.
CLASSIFICATION_METHODS = {
    "br": (MLC.BRClassifier, {}, {}),
    "clr": (MLC.CLRClassifier, {}, {}),
    "cc": (MLC.CCClassifier, {"order": None, "random_state": RANDOM_STATE}, {}),
    "lp": (MLC.LPClassifier, {}, {}),
    "pst": (
        MLC.PStClassifier,
        {},
        {"p": ("pruning_value", [25, 50, 100]), "b": ("max_reintroduced", [1, 2])},
    ),
    "mbr": (MLC.MBRClassifier, {"cv": 5, "random_state": RANDOM_STATE}, {}),
    "rakel": (MLC.RAkELClassifier, {"random_state": RANDOM_STATE}, {}),
    "homer": (MLC.HOMERClassifier, {"random_state": RANDOM_STATE}, {}),
}

CLASSIFICATION_ALGORITHM = {
    # Explicit one-vs-rest: reproduces liblinear's multiclass semantics (LP/PSt/RAkEL/HOMER)
    # independently of the scikit-learn version, so the candidate set does not depend on the environment.
    "lr": OneVsRestClassifier(
        LogisticRegression(solver="liblinear", max_iter=10000, random_state=RANDOM_STATE)
    ),
    "gnb": GaussianNB(),
    "dt": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "rf": RandomForestClassifier(random_state=RANDOM_STATE),
}

EMBEDDING_SOURCES = [
    {
        "name": "GloVe.6B.50D",
        "model-path": os.getenv("GLOVE_6B_50D_PATH"),
        "embedding-dim": 50,
    },
    {
        "name": "GloVe.6B.100D",
        "model-path": os.getenv("GLOVE_6B_100D_PATH"),
        "embedding-dim": 100,
    },
    {
        "name": "GloVe.6B.200D",
        "model-path": os.getenv("GLOVE_6B_200D_PATH"),
        "embedding-dim": 200,
    },
    {
        "name": "GloVe.6B.300D",
        "model-path": os.getenv("GLOVE_6B_300D_PATH"),
        "embedding-dim": 300,
    },
    {
        "name": "DistilRoBERTa",
        "model-path": os.getenv("DISTILROBERTA_PATH"),
        "embedding-dim": None,
    },
    {"name": "SBERT", "model-path": os.getenv("SBERT_PATH"), "embedding-dim": None},
    {
        "name": "ATT&CK-BERT",
        "model-path": os.getenv("ATTACK_BERT_PATH"),
        "embedding-dim": None,
    },
]

SYSTEM_MODULES = [
    {
        "level": "general",
        "targets": {
            "computer security": "computer security",
            "operating systems": "operating systems",
            "software": "software",
            "programming languages": "software",
            "hardware": "hardware",
            "electronic components": "hardware",
            "networking": "networking",
            "internet technology": "networking",
        },
    },
    {
        "level": "computer security",
        "targets": {
            "network security": "network security",
            "antivirus and malware": "antivirus and malware",
        },
    },
    {
        "level": "operating systems",
        "targets": {
            "mac os": "mac os",
            "windows": "windows",
            "unix": "unix",
            "linux": "linux",
        },
    },
    {"level": "software", "targets": {"databases": "databases"}},
    {
        "level": "hardware",
        "targets": {
            "computer": "computer",
            "computer components": "computer components",
            "computer networking": "computer networking",
        },
    },
]

In [ ]:
# Local-only tiny-config mechanism (NON-PRODUCTION).  The subset/subsample
# variables below are honoured only with MLC_LOCAL_TINY=1; the selection is
# recorded in the run identity (RUN_CONFIG) and printed as "NON-PRODUCTION
# CONFIG" in the summary.  Whenever assembly runs (stage assemble or all),
# any such variable without MLC_LOCAL_TINY=1, and MLC_UNITS in any case, is
# rejected: the System must cover every module x embedder x candidate.
#   MLC_METHODS / MLC_ALGORITHMS: comma-separated keys to keep
#   MLC_EMBEDDERS: comma-separated embedder names to keep
#   MLC_SMOKE_ROWS / MLC_SMOKE_TEST_ROWS: deterministic train/test subsample
ALLOW_FAILED_CANDIDATES = os.getenv("ALLOW_FAILED_CANDIDATES", "").strip() == "1"  # assemble only
_TINY_VARS = ("MLC_METHODS", "MLC_ALGORITHMS", "MLC_EMBEDDERS", "MLC_SMOKE_ROWS", "MLC_SMOKE_TEST_ROWS")
LOCAL_TINY = os.getenv("MLC_LOCAL_TINY", "").strip() == "1"
_set = [k for k in _TINY_VARS if os.getenv(k, "").strip()]
if STAGE in ("assemble", "all"):
    if os.getenv("MLC_UNITS", "").strip():
        raise RuntimeError(f"MLC_STAGE={STAGE} assembles the full configuration; MLC_UNITS is only allowed for MLC_STAGE=train.")
    if _set and not LOCAL_TINY:
        raise RuntimeError(f"MLC_STAGE={STAGE} assembles the full configuration; subset override(s) {_set} are only allowed with the local-only MLC_LOCAL_TINY=1.")


def _subset(env_name, available):
    spec = os.getenv(env_name, "").strip()
    if not spec:
        return list(available)
    keep = [s.strip() for s in spec.split(",") if s.strip()]
    unknown = sorted(set(keep) - set(available))
    if unknown:
        raise ValueError(f"{env_name}: unknown {unknown}; expected some of {list(available)}")
    return [a for a in available if a in keep]


CLASSIFICATION_METHODS = {
    k: CLASSIFICATION_METHODS[k] for k in _subset("MLC_METHODS", CLASSIFICATION_METHODS)
}
CLASSIFICATION_ALGORITHM = {
    k: CLASSIFICATION_ALGORITHM[k]
    for k in _subset("MLC_ALGORITHMS", CLASSIFICATION_ALGORITHM)
}
_embedders = _subset("MLC_EMBEDDERS", [e["name"] for e in EMBEDDING_SOURCES])
EMBEDDING_SOURCES = [e for e in EMBEDDING_SOURCES if e["name"] in _embedders]
SMOKE_ROWS = int(os.getenv("MLC_SMOKE_ROWS", "0") or 0)  # 0 = all rows
SMOKE_TEST_ROWS = int(os.getenv("MLC_SMOKE_TEST_ROWS", "0") or 0)  # 0 = all rows
NON_PRODUCTION = {k: os.environ[k] for k in _TINY_VARS if os.getenv(k, "").strip()}
if NON_PRODUCTION:
    NON_PRODUCTION["MLC_LOCAL_TINY"] = "1" if LOCAL_TINY else "0"
    print(f"*** NON-PRODUCTION CONFIG (local check only, not a full run): {NON_PRODUCTION} ***")

# Output directories of the assemble stage.  A non-production configuration must never
# overwrite the production files: it requires MODELS_DIR and MAPPINGS_DIR to differ from
# the defaults (checked here, before any write, including checkpoints).
DEFAULT_MODELS_DIR, DEFAULT_MAPPINGS_DIR = "models", "mappings"
MODELS_DIR = os.getenv("MODELS_DIR", "").strip() or DEFAULT_MODELS_DIR
MAPPINGS_DIR = os.getenv("MAPPINGS_DIR", "").strip() or DEFAULT_MAPPINGS_DIR
if NON_PRODUCTION and STAGE in ("assemble", "all"):
    for _name, _dir, _default in (
        ("MODELS_DIR", MODELS_DIR, DEFAULT_MODELS_DIR),
        ("MAPPINGS_DIR", MAPPINGS_DIR, DEFAULT_MAPPINGS_DIR),
    ):
        if Path(_dir).resolve() == Path(_default).resolve():
            raise RuntimeError(
                f"NON-PRODUCTION config {NON_PRODUCTION} refuses the default {_name}={_default!r}: "
                f"set {_name} to a different directory (checked before any write)."
            )

# Candidates: methods (grid points in order) x algorithms, as product(methods, algorithms).
def build_candidates(methods, algorithms):
    candidates = []
    for mtd, (cls, fixed, grid) in methods.items():
        points = [{}]
        for short, (param, values) in grid.items():
            points = [{**p, short: v} for p in points for v in values]
        for point in points:
            label = mtd.upper()
            if point:
                label += "(" + ",".join(f"{s}={v}" for s, v in point.items()) + ")"
            params = {**fixed, **{grid[s][0]: v for s, v in point.items()}}
            for alg in algorithms:
                candidates.append(
                    {"name": f"{label}[{alg.upper()}]", "method": mtd, "params": params, "alg": alg}
                )
    return candidates


CANDIDATES = build_candidates(CLASSIFICATION_METHODS, CLASSIFICATION_ALGORITHM)
assert len({c["name"] for c in CANDIDATES}) == len(CANDIDATES)
assert all(" - " not in c["name"] for c in CANDIDATES)
print(f"{len(CANDIDATES)} candidates per module x embedder")

## Data

In [ ]:
def load_train_frame(path):
    """Training CSV with parsed columns, cleaned text and the 'other' relabelling of
    irrelevant tweets."""
    with open(path, "rb") as f:
        frame = pd.read_csv(f)
    frame = frame.assign(
        tweet=lambda df: df["tweet"].apply(func=ast.literal_eval),
        watson=lambda df: df["watson"].apply(func=ast.literal_eval),
        urls=lambda df: df["urls"].apply(func=ast.literal_eval),
        watson_list=lambda df: df["watson_list"].apply(func=ast.literal_eval),
        target=lambda df: df["target"].apply(func=ast.literal_eval),
    )
    frame["text"] = [su.replace_txt_components(t) for t in frame["text"]]
    for idx, relevant in enumerate(frame["relevant"]):
        if not relevant and "other" not in frame.loc[idx, "target"]:
            frame.loc[idx, "target"] = ["other"]
    return frame


def smoke_sample(frame, rows, random_state, levels):
    """Smoke only: deterministic, roughly proportional sample of each group
    (labelset, module membership) with >= 5 rows, at least 5 rows per group,
    so every module keeps all its classes and can be split stratified."""
    shuffled = frame.sample(frac=1.0, random_state=random_state)
    group = pd.Series(
        [
            (tuple(t), tuple(level in w for level in levels))
            for t, w in zip(shuffled["target"], shuffled["watson_list"])
        ],
        index=shuffled.index,
    )
    keep = []
    for _, grp in shuffled.groupby(group, sort=False):
        if len(grp) >= 5:
            keep.extend(grp.index[: max(5, round(len(grp) * rows / len(frame)))])
    return frame.loc[sorted(keep)].reset_index(drop=True)


train = load_train_frame(os.getenv("ONE_STEP_CATEGORIES_TRAIN_CSV"))
if SMOKE_ROWS:
    train = smoke_sample(train, SMOKE_ROWS, RANDOM_STATE, [m["level"] for m in SYSTEM_MODULES])

def train_texts(frame):
    """Texts embedded for training: the replacement is applied once more on top of
    load_train_frame (twice in total, as the original loop did for every module
    subset), so the cached embeddings use the same texts."""
    return pd.Series([su.replace_txt_components(t) for t in frame["text"]])


TRAIN_TEXTS = train_texts(train)
print(f"Training tweets: {len(train)}")

In [7]:
with open(os.getenv("ONE_STEP_CATEGORIES_TEST_CSV"), "rb") as f:
    test_tweets = pd.read_csv(f)
    test_tweets = test_tweets.assign(
        tweet=lambda df: df["tweet"].apply(func=ast.literal_eval),
        watson=lambda df: df["watson"].apply(func=ast.literal_eval),
        urls=lambda df: df["urls"].apply(func=ast.literal_eval),
        watson_list=lambda df: df["watson_list"].apply(func=ast.literal_eval),
        target=lambda df: df["target"].apply(func=ast.literal_eval),
    )
    test_tweets["text"] = [su.replace_txt_components(t) for t in test_tweets["text"]]
    relevant = test_tweets["relevant"] == True
    is_other = test_tweets["target"].apply(lambda x: "other" in x)
    for idx in test_tweets[relevant & is_other].index:
        test_tweets.loc[idx, "relevant"] = False

if SMOKE_TEST_ROWS:  # smoke only: deterministic sample
    test_tweets = test_tweets.sample(
        n=min(SMOKE_TEST_ROWS, len(test_tweets)), random_state=RANDOM_STATE
    ).sort_index().reset_index(drop=True)

texts = test_tweets["text"]
targets = test_tweets["target"]
print(f"Threat Tweets: {len(test_tweets)}")
test_tweets.head()

Threat Tweets: 4067


,Unnamed: 0,_id,date,id,relevant,text,tweet,type,watson,annotation,urls,destination_url,valid_certificate,watson_list,target
0,1533,b'5b8e2e39bb325e2996c66aab',2018-09-04 07:03:16+00:00,1036872307057352704,True,threatmeter: irisgraphic sql injection vulnera...,{'created_at': 'Tue Sep 04 07:03:16 +0000 2018...,vulnerability,{'technology and computing': {'computer securi...,threat,[https://ift.tt/2wNsX91],https://cxsecurity.com/issue/WLB-2018090028,True,"[technology and computing, computer security, ...","[antivirus and malware, software, databases, c..."
1,59,b'5b887f63bb325e65fa7e7922',2018-08-30 23:36:02+00:00,1035310204102078464,False,Download this how-to guide on preparing for an...,{'created_at': 'Thu Aug 30 23:36:02 +0000 2018...,ransomware,{'technology and computing': {'computer securi...,business,[http://bit.ly/2wuUTyT],https://logrhythm.com/the-ransomware-threat-eb...,True,"[technology and computing, computer security, ...","[antivirus and malware, computer security]"
2,10824,b'5b92cad0bb325e521c5ae40a',2018-09-07 19:00:31+00:00,1038139971562078208,True,The US Justice Department is poised to charge ...,{'created_at': 'Fri Sep 07 19:00:31 +0000 2018...,ransomware,"{'law, govt and politics': {'government': {}, ...",threat,[https://twitter.com/i/web/status/103813997156...,https://twitter.com/i/web/status/1038139971562...,True,"[law, govt and politics, government, armed for...","[antivirus and malware, computer security]"
3,2104,b'5b90af51bb325e70071868e2',2018-09-06 04:38:41+00:00,1037560694789554176,False,"3) Control and monitor compliance. Also, in...",{'created_at': 'Thu Sep 06 04:38:41 +0000 2018...,general,{'finance': {'personal finance': {'insurance':...,NaN,[https://twitter.com/i/web/status/103756069478...,https://twitter.com/i/web/status/1037560694789...,True,"[finance, personal finance, insurance, busines...",[other]
4,12292,b'5b935e61bb325e521c5aeab9',2018-09-08 05:30:09+00:00,1038298421893967877,True,"Ransomware down, but not out, report reveals",{'created_at': 'Sat Sep 08 05:30:09 +0000 2018...,ransomware,{'business and industrial': {'company': {'annu...,threat,[http://fsecure.smh.re/4AS],http://fsecure.smh.re/4AS,True,"[business and industrial, company, annual repo...","[databases, software]"


## Run identity and checkpoints

Stages: `MLC_STAGE` = `embed`, `train`, `assemble` or `all` (default; the three in one session). Checkpoints go to `CHECKPOINT_DIR` (default `./checkpoints`); earlier stages are read from `CHECKPOINT_INPUTS`. Training units: `MLC_UNITS` (`<level>:<embedder name>`, comma-separated; default all module x embedder pairs; train only). Candidates that fail to fit with an expected error are stored as failure markers; assemble fails unless `ALLOW_FAILED_CANDIDATES=1`. Local-only tiny runs use `MLC_LOCAL_TINY=1` with `MLC_METHODS`, `MLC_ALGORITHMS`, `MLC_EMBEDDERS`, `MLC_SMOKE_ROWS`, `MLC_SMOKE_TEST_ROWS`; they are recorded in the run identity and flagged NON-PRODUCTION CONFIG; with such a configuration the assemble stage requires `MODELS_DIR` and `MAPPINGS_DIR` to differ from the defaults (`models`, `mappings`) and refuses otherwise, before any write. Each embedder's input is fingerprinted (sha256 of the GloVe file or model directory, else the Hugging Face commit of the snapshot loaded; `unresolved` if it cannot be determined) and stored with its embeddings; train and assemble verify it, and both train (before any fit) and assemble refuse `unresolved` unless `ALLOW_UNRESOLVED_EMBEDDERS=1`. The run identity hashes the source of the cell-defined functions that shape training outputs (data loading and relabelling, smoke sampling, subset and candidate construction, embedder helpers, unit split, training loop) with `inspect.getsource`, so editing them starts a new identity, as does any change of the final training texts (their sha256 is part of the identity, which covers `notebooks/utils.py` replacements and the data); the effective training data is part of it too: per module the sha256 of the training rows (positions and texts), the binarized label matrix (bytes, dtype, shape), the class order and the train/validation split indices (derived from data and hashed code only, identical in every stage); assembly and evaluation code (including the `ModuleSystem` and `System` classes, which training does not use) is not part of it. Instead the assemble stage computes a provenance digest (printed as `RESULT/PROVENANCE: <sha256>` and stored in the `metadata` of the saved modules, System and `mappings/*.pickle.meta.json`) over the run identity, the sha256 of the assembly cell text (IPython input history, fails closed) and of the `ModuleSystem`/`System` method sources, the sidecar sha256 of every checkpoint (all candidates, hence every selected one, plus embeddings and fingerprints), the validation split, the sha256 of the test texts, targets and relevance flags, and the infeasible and excluded candidates; the metadata also records the training and test text digests separately. A candidate whose documented data precondition cannot hold for its training labels (PSt with `pruning_value` above the largest labelset count; MBR with fewer rows than `cv`) is recorded as *infeasible* instead of being fitted; unlike a failure it does not block assembly and is listed in its output.

In [ ]:
# Checkpoint store shared by all stages.  CHECKPOINT_DIR is this session's
# output directory; CHECKPOINT_INPUTS lists read-only outputs of earlier stages.
def _jsonable(value):
    if value is None or isinstance(value, (str, bool, int, float)):
        return value
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in value.items()}
    return repr(value)


def text_digest(texts) -> str:
    return hashlib.sha256(json.dumps(list(texts), ensure_ascii=True).encode()).hexdigest()


SPLIT_TEXTS = {"train": TRAIN_TEXTS, "test": texts}
SPLIT_DIGEST = {split: text_digest(t) for split, t in SPLIT_TEXTS.items()}


def embedding_key(embedder_name: str, split: str) -> tuple:
    return ("embedding", embedder_name, split, SPLIT_DIGEST[split])


def fingerprint_key(embedder_name: str) -> tuple:
    """Content identity of the embedder input the embeddings above were computed from
    (written by the embed stage after the embeddings, verified by train and assemble)."""
    return ("fingerprint", embedder_name)


def candidate_key(level: str, embedder_name: str, candidate_name: str) -> tuple:
    return ("candidate", level, embedder_name, candidate_name)


_FP_CACHE = {}  # per session: the sha256 of a big embedder file/directory is computed once


def _dir_sha256(path) -> str:
    root = Path(path)
    h = hashlib.sha256()
    for p in sorted(q for q in root.rglob("*") if q.is_file()):
        rel = p.relative_to(root)
        if any(part.startswith(".") for part in rel.parts):
            continue
        h.update(rel.as_posix().encode() + b"\0" + ck.file_sha256(p).encode() + b"\0")
    return h.hexdigest()


def _hf_revision(model_id, online=True):
    """Commit hash of a Hugging Face model: the snapshot in the local cache (the one
    SentenceTransformer actually loaded), else with ``online`` the hub's current commit;
    None if unresolvable."""
    try:
        from huggingface_hub import snapshot_download

        return Path(snapshot_download(model_id, local_files_only=True)).name
    except Exception:
        pass
    if online:
        try:
            from huggingface_hub import model_info

            return model_info(model_id, timeout=10).sha
        except Exception:
            pass
    return None


def embedder_fingerprint(source, online=True):
    """Content identity of an embedder input: sha256 of the GloVe file (or model
    directory), else the Hugging Face commit hash.  The path is deliberately not part of
    it (the same content may live under another path); an unresolvable HF revision is
    recorded as 'unresolved'.  None when a file/directory input is not available here."""
    path = source["model-path"]
    if source["embedding-dim"] is not None or (path and os.path.exists(path)):
        if not path or not os.path.exists(path):
            return None
        if path not in _FP_CACHE:
            if os.path.isdir(path):
                _FP_CACHE[path] = {"kind": "dir", "sha256": _dir_sha256(path)}
            else:
                _FP_CACHE[path] = {"kind": "file", "sha256": ck.file_sha256(path)}
        return _FP_CACHE[path]
    revision = _hf_revision(path, online=online)
    if revision is None:
        print(f"WARNING: cannot resolve the revision of {path}; recorded as 'unresolved'.")
    return {"kind": "hf", "model": path, "revision": revision or "unresolved"}


def same_input(a, b):
    """True/False when both fingerprints are resolved (GloVe/directory: sha256; Hugging Face: hub id
    and revision), None if one of them is 'unresolved' (cannot be told)."""
    if a is None or b is None:
        return None
    if a["kind"] != b["kind"]:
        return False
    if a["kind"] == "hf" and a["model"] != b["model"]:  # another hub id: a different model whatever the revision
        return False
    field = "revision" if a["kind"] == "hf" else "sha256"
    if "unresolved" in (a[field], b[field]):
        return None
    return a[field] == b[field]


def allow_unresolved_embedders() -> bool:
    return os.getenv("ALLOW_UNRESOLVED_EMBEDDERS", "").strip() == "1"  # knowing opt-out; not part of the identity


def check_embedder_input(name, stored, current, require=False):
    """Stage guard: the embeddings cached for ``name`` came from ``stored``; refuse when the
    embedder input available now is demonstrably different.  With ``require`` (train, assemble) an
    input that is not available here (current None) is refused unless ALLOW_UNRESOLVED_EMBEDDERS=1."""
    verdict = same_input(stored, current)
    if current is None and require and not allow_unresolved_embedders():
        raise RuntimeError(
            f"Embedder input of {name} is not available here, so the cached embeddings cannot be verified "
            f"(stored {stored}). Mount the same input, or set ALLOW_UNRESOLVED_EMBEDDERS=1 to accept knowingly."
        )
    if current is None:
        print(f"WARNING: embedder input of {name} not available here; cached embeddings taken on trust of {stored}.")
    elif verdict is False:
        raise RuntimeError(
            f"Embedder input of {name} changed since the embeddings were cached: stored {stored} != current {current}. "
            "Cached embeddings are not reused; start from a new CHECKPOINT_DIR."
        )
    elif verdict is None:
        print(f"WARNING: cannot verify the embedder input of {name} ({stored} vs {current}).")


def build_candidate(candidate):
    cls = CLASSIFICATION_METHODS[candidate["method"]][0]
    return cls(clone(CLASSIFICATION_ALGORITHM[candidate["alg"]]), **candidate["params"])


def infeasible_reason(candidate, y_train):
    """Why ``candidate`` cannot be fitted on these training labels by design, or None.
    Only data preconditions documented by the MLC methods are checked, decided from the
    labels alone, before fitting: PSt needs a labelset occurring at least pruning_value
    times (PStClassifier.fit raises ValueError otherwise); MBR needs at least cv samples.
    Every other fit error remains a (blocking) failure."""
    y_train = np.asarray(y_train)
    if candidate["method"] == "pst":
        _, counts = np.unique(y_train.astype(int), axis=0, return_counts=True)
        p = candidate["params"]["pruning_value"]
        if counts.max() < p:
            return f"max labelset count {counts.max()} < pruning_value {p} in the {len(y_train)} training rows"
    elif candidate["method"] == "mbr":
        cv = candidate["params"].get("cv")
        if cv is not None and len(y_train) < cv:
            return f"{len(y_train)} training rows < cv {cv}"
    return None


def _array_digest(a) -> str:
    """sha256 of an array bound to its dtype and shape (not only its bytes)."""
    a = np.ascontiguousarray(a)
    h = hashlib.sha256(f"{a.dtype.str}|{a.shape}|".encode())
    h.update(a.tobytes())
    return h.hexdigest()


def unit_split(module_config):
    """Rows of a module, its binarized targets (columns in ``mlb.classes_`` order) and the
    stratified train/validation split of the original loop as index arrays into those rows.
    Depends on the training data only (no embeddings), so the run identity can bind it."""
    tweets = train.assign(
        target=lambda df: df["target"].apply(
            func=su.translate_src_categories, args=(module_config["targets"],)
        )
    )
    if module_config["level"] != "general":
        rows = tweets["watson_list"].apply(lambda x: module_config["level"] in x).to_numpy()
    else:
        rows = np.ones(len(tweets), dtype=bool)
    targets_ = tweets["target"][rows].reset_index(drop=True)
    mlb = preprocessing.MultiLabelBinarizer()
    y = mlb.fit_transform(y=targets_)
    _, y_mcp = np.unique(ar=y, axis=0, return_inverse=True)
    idx_train, idx_val = train_test_split(
        np.arange(len(y)),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
        stratify=y_mcp,
    )
    return rows, mlb, y, idx_train, idx_val


def unit_data(module_config, X_all):
    """Rows, targets and the stratified train/validation split of a module
    (as in the original loop); X_all = embeddings of TRAIN_TEXTS."""
    rows, mlb, y, idx_train, idx_val = unit_split(module_config)
    x = np.asarray(X_all)[rows]
    return mlb, x[idx_train], x[idx_val], y[idx_train], y[idx_val]


def effective_data(modules, train_texts_list):
    """Digest of the data each module is actually fitted and selected on: its training rows
    (row positions and texts), the binarized label matrix (bytes, dtype, shape), the class
    order (``mlb.classes_``) and the train/validation split indices.  Per module and as a
    whole (``sha256``).  Derived from data and hashed code only, never from stage settings."""
    out = {}
    for m in modules:
        rows, mlb, y, idx_train, idx_val = unit_split(m)
        row_ids = np.flatnonzero(rows)
        part = {
            "n_rows": int(len(row_ids)),
            "row_ids_sha256": _array_digest(row_ids),
            "texts_sha256": text_digest([train_texts_list[i] for i in row_ids]),
            "labels_sha256": _array_digest(y),
            "classes": [str(c) for c in mlb.classes_],
            "train_idx_sha256": _array_digest(idx_train),
            "val_idx_sha256": _array_digest(idx_val),
        }
        part["sha256"] = hashlib.sha256(json.dumps(part, sort_keys=True).encode()).hexdigest()
        out[m["level"]] = part
    out["sha256"] = hashlib.sha256(
        json.dumps({k: v["sha256"] for k, v in out.items()}, sort_keys=True).encode()
    ).hexdigest()
    return out


# Deterministic fit failures of a candidate are stored as markers that assemble reports and
# refuses (unless ALLOW_FAILED_CANDIDATES=1).  Anything else (MemoryError, KeyboardInterrupt,
# SystemExit, OSError, ...) propagates before anything is saved, so the candidate is retried.
EXPECTED_FIT_ERRORS = (ValueError, TypeError, np.linalg.LinAlgError, NotFittedError)


def train_units(store, budget, units, candidates, module_by_level, embedder_fp):
    """The training loop (a function defined before the run identity so that its source is
    part of it).  One checkpoint per (level, embedder, candidate): fitted model and validation
    assessment, an 'infeasible' record (data precondition cannot hold, see infeasible_reason)
    or a failure marker.  Verified checkpoints are skipped; the budget stops before a candidate
    whose estimated duration does not fit.  Returns (done, missing, failed, infeasible)."""
    done, missing, failed, infeasible = [], [], {}, {}
    stop = False
    for level, embedder_name in tqdm(units, desc="Units", unit="unit"):
        split = None  # built on the first candidate that needs fitting
        for candidate in candidates:
            key = candidate_key(level, embedder_name, candidate["name"])
            if stop:
                missing.append(key)
                continue
            if store.has(key):
                done.append(key)
                continue
            if split is None:
                split = unit_data(module_by_level[level], store.load(embedding_key(embedder_name, "train")))
            mlb, X_train, X_val, y_train, y_val = split
            why = infeasible_reason(candidate, y_train)
            if why is not None:  # by design, decided from the labels: nothing is fitted
                store.save(key, {"model": None, "assess": None, "error": None, "infeasible": why,
                                 "embedder_input": embedder_fp[embedder_name]})
                infeasible[key] = why
                done.append(key)
                continue
            if not budget.allows(None, kind=candidate["name"]):
                print(f"INCOMPLETE: budget exhausted before {key}")
                stop = True
                missing.append(key)
                continue
            t0 = time.monotonic()
            model = build_candidate(candidate)
            try:
                model.fit(X_train, y_train)
            except EXPECTED_FIT_ERRORS as e:
                print(f"{candidate['name']} - {embedder_name} ({level}) failed to fit due to {e}")
                record = {"model": None, "assess": None, "error": f"{type(e).__name__}: {e}",
                          "infeasible": None, "embedder_input": embedder_fp[embedder_name]}
                failed[key] = record["error"]
            else:
                record = {"model": model, "assess": model.evaluate(X_val, y_val), "error": None,
                          "infeasible": None, "embedder_input": embedder_fp[embedder_name]}
            store.save(key, record)
            budget.record(candidate["name"], time.monotonic() - t0)
            done.append(key)
    return done, missing, failed, infeasible


# Run identity: the source of every cell-defined function that shapes training outputs (data
# loading and 'other' relabelling, smoke sampling, subset selection, candidate construction,
# the embedder helpers, unit_split/unit_data/effective_data, the training loop with its fit-error and infeasibility
# logic) is hashed with inspect.getsource.  Why functions and not cell texts: on Kaggle the
# kernel runs a copy of the notebook, so cell texts are not reliably available, whereas
# IPython registers the source of every function it executes (the same mechanism that
# already served the previous hash).  Assembly and evaluation code is deliberately NOT
# covered (ModuleSystem and System are built only at assembly; checkpoints hold fitted
# candidates, never these classes): an assembly-only fix must not invalidate training
# checkpoints.  The final training texts are covered by their digest in RUN_CONFIG.  Nothing here
# depends on the stage (stage values come from env and select cells, not code).
HASHED_CODE = (
    _subset, build_candidates, load_train_frame, smoke_sample, train_texts,
    text_digest, embedding_key, fingerprint_key, candidate_key, _dir_sha256, _hf_revision,
    embedder_fingerprint, same_input, build_candidate, infeasible_reason,
    _array_digest, unit_split, unit_data, effective_data, train_units,
)
CODE_SOURCE_SHA256 = {obj.__name__: hashlib.sha256(inspect.getsource(obj).encode("utf-8")).hexdigest()
                      for obj in HASHED_CODE}
assert len(CODE_SOURCE_SHA256) == len(HASHED_CODE) and all(
    inspect.getsource(obj).lstrip().startswith("def ") for obj in HASHED_CODE
), "run identity: code coverage is empty or a source lookup failed"


EFFECTIVE_DATA = effective_data(SYSTEM_MODULES, list(TRAIN_TEXTS))
print(f"Effective data digest: {EFFECTIVE_DATA['sha256']}")


RUN_CONFIG = {
    "config_cell_sha256": hashlib.sha256(CONFIG_CELL_SOURCE.encode("utf-8")).hexdigest(),
    # cell-defined code that shapes the cached embeddings, splits, candidates and outputs
    # (the cleaning/embedding code itself lives in notebooks/utils.py, hashed below)
    "cell_source_sha256": hashlib.sha256("\n".join(f"{k}={v}" for k, v in CODE_SOURCE_SHA256.items()).encode("utf-8")).hexdigest(),
    "cell_sources": CODE_SOURCE_SHA256,
    # digest of the final training texts (after load_train_frame, train_texts and notebooks.utils),
    # so a change in the replacement code of notebooks.utils or in the data is covered directly
    "train_texts_sha256": SPLIT_DIGEST["train"],
    # per module: training rows + binarized labels + class order + train/validation split indices
    "effective_data": EFFECTIVE_DATA,
    "expected_fit_errors": [e.__name__ for e in EXPECTED_FIT_ERRORS],
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "methods": {
        k: {"class": cls.__name__, "params": _jsonable(fixed), "grid": _jsonable(grid)}
        for k, (cls, fixed, grid) in CLASSIFICATION_METHODS.items()
    },
    "algorithms": {
        k: {"class": type(alg).__name__, "params": _jsonable(alg.get_params())}
        for k, alg in CLASSIFICATION_ALGORITHM.items()
    },
    "candidates": [c["name"] for c in CANDIDATES],
    # name and dimension only: model-path is a mount location (it differs between chained Kaggle
    # kernels), not content; the embedder content is bound by the per-embedder fingerprints
    "embedders": [{k: v for k, v in _jsonable(e).items() if k != "model-path"} for e in EMBEDDING_SOURCES],
    "modules": _jsonable(SYSTEM_MODULES),
    "non_production": NON_PRODUCTION,
    "smoke_rows": SMOKE_ROWS,
    "smoke_test_rows": SMOKE_TEST_ROWS,
}
IDENTITY = ck.make_identity(
    code_paths={
        **{p: p for p in sorted(glob.glob("MLC/*.py"))},
        "notebooks/utils.py": "notebooks/utils.py",
        "notebooks/checkpoints.py": "notebooks/checkpoints.py",
    },
    data_paths={
        "train": os.getenv("ONE_STEP_CATEGORIES_TRAIN_CSV"),
        "test": os.getenv("ONE_STEP_CATEGORIES_TEST_CSV"),
    },
    config=RUN_CONFIG,
    # per-embedder inputs (file sha256 / HF commit) are stored with the embeddings: ("fingerprint", name)
    modules=["scikit-learn", "numpy", "sentence-transformers", "transformers", "torch"],
)
store = ck.Store.from_env(IDENTITY)
print(f"Checkpoints: {store.out_dir} (inputs: {[str(d) for d in store.search_dirs]})")
print(f"Run identity: {store.digest}")


ALL_UNITS = [(m["level"], e["name"]) for m in SYSTEM_MODULES for e in EMBEDDING_SOURCES]
_units_spec = os.getenv("MLC_UNITS", "").strip()
if _units_spec:
    _wanted = [tuple(u.strip().split(":", 1)) for u in _units_spec.split(",") if u.strip()]
    _bad = [u for u in _wanted if u not in ALL_UNITS]
    if _bad:
        raise ValueError(f"MLC_UNITS: unknown units {_bad}; expected '<level>:<embedder name>' from {ALL_UNITS}")
    UNITS = [u for u in ALL_UNITS if u in _wanted]
else:
    UNITS = list(ALL_UNITS)
EXPECTED_KEYS = [
    embedding_key(e["name"], split) for e in EMBEDDING_SOURCES for split in SPLIT_TEXTS
] + [fingerprint_key(e["name"]) for e in EMBEDDING_SOURCES] + [
    candidate_key(level, name, c["name"]) for level, name in ALL_UNITS for c in CANDIDATES
]
MODULE_BY_LEVEL = {m["level"]: m for m in SYSTEM_MODULES}
SOURCE_BY_NAME = {e["name"]: e for e in EMBEDDING_SOURCES}

## Stage: embed

Embeddings of all training and all test texts for every embedder, computed once and cached (`npy`, keyed by embedder, split and the digest of the text list).

In [ ]:
if STAGE in ("embed", "all"):
    _done = []
    for source in tqdm(EMBEDDING_SOURCES, desc="Embedders", unit="embedder"):
        name = source["name"]
        fp_key = fingerprint_key(name)
        stored = store.load(fp_key) if store.has(fp_key) else None
        # Cached embeddings are trusted only together with their stored fingerprint.  Without
        # it (first run, or an interrupted one) every split is recomputed; a cache entry that
        # differs then raises ConflictError at save instead of being reused.
        todo = [s for s in SPLIT_TEXTS if stored is None or not store.has(embedding_key(name, s))]
        fresh = {}
        for split in todo:
            if not source["model-path"]:
                raise ValueError(f"{name}: model path is not set (see EMBEDDING_SOURCES)")
            fresh[split] = np.asarray(
                su.preprocess_texts(
                    list_str=SPLIT_TEXTS[split],
                    model_path=source["model-path"],
                    embedding_dim=source["embedding-dim"],
                )
            )
        # After embedding: for Hugging Face models the local cache then holds the snapshot actually loaded.
        current = embedder_fingerprint(source)
        if stored is not None:
            check_embedder_input(name, stored, current)
        for split, X_split in fresh.items():
            store.save(embedding_key(name, split), X_split, fmt="npy")
            print(f"embedded {name} {split}: {X_split.shape}")
        if stored is None:
            store.save(fp_key, current)  # last: marks the embeddings of this embedder as verified
            stored = current
        if "unresolved" in stored.values():
            print(f"WARNING: embedder input of {name} is unresolved; train and assemble refuse it unless ALLOW_UNRESOLVED_EMBEDDERS=1.")
        _done += [embedding_key(name, split) for split in SPLIT_TEXTS] + [fp_key]
    ck.write_status(store.out_dir, "COMPLETE", done=_done, identity_digest=store.digest)
    print(f"embed: COMPLETE ({len(_done)} artifacts)")
else:
    print(f"embed: skipped (MLC_STAGE={STAGE})")

## Model Training (stage: train)

Per module: the stratified train/validation split of the original loop (on cached embeddings only), then every candidate (method x algorithm) is fitted and assessed on validation.

In [ ]:
# One checkpoint per candidate (level, embedder, candidate): fitted model and
# validation assessment, or the fit error.  Verified checkpoints are skipped;
# the budget (CHECKPOINT_BUDGET_HOURS etc.) stops before a candidate whose
# estimated duration (longest earlier fit of that candidate) does not fit.
# A candidate whose documented data precondition cannot hold for its training labels
# (infeasible_reason: e.g. PSt pruning_value above the largest labelset count) is stored as an
# 'infeasible' record, distinct from a fit failure: it does not block assemble.
TRAIN_INCOMPLETE = False
if STAGE in ("train", "all"):
    budget = ck.Budget.from_env()
    _consumed = sorted({name for _, name in UNITS})
    ck.verify_complete(  # fail closed: training only consumes cached embeddings and their fingerprints
        store,
        [embedding_key(name, "train") for name in _consumed] + [fingerprint_key(name) for name in _consumed],
    )
    EMBEDDER_FP = {name: store.load(fingerprint_key(name)) for name in _consumed}
    for name, stored in EMBEDDER_FP.items():  # the embedder input must still be the one embedded
        check_embedder_input(name, stored, embedder_fingerprint(SOURCE_BY_NAME[name], online=False), require=True)
    _unresolved = [n for n, fp in EMBEDDER_FP.items() if fp is not None and "unresolved" in fp.values()]
    if _unresolved and not allow_unresolved_embedders():
        raise RuntimeError(  # before any fit: an unverifiable embedder input would make the models unattributable
            f"Embedder input(s) {_unresolved} have an 'unresolved' fingerprint: refusing to train before any fit. "
            "Re-run the embed stage with network access to the Hugging Face hub, or set ALLOW_UNRESOLVED_EMBEDDERS=1 to accept."
        )
    _done, _missing, _failed, _infeasible = train_units(
        store, budget, UNITS, CANDIDATES, MODULE_BY_LEVEL, EMBEDDER_FP
    )
    TRAIN_INCOMPLETE = bool(_missing)
    ck.write_status(
        store.out_dir,
        "INCOMPLETE" if TRAIN_INCOMPLETE else "COMPLETE",
        done=_done,
        missing=_missing,
        failed=_failed,
        identity_digest=store.digest,
    )
    for _k, _msg in _failed.items():
        print(f"!!! FAILED CANDIDATE {_k[1]} / {_k[2]} / {_k[3]}: {_msg}")
    for _k, _msg in _infeasible.items():
        print(f"infeasible by design (not an error) {_k[1]} / {_k[2]} / {_k[3]}: {_msg}")
    print(
        f"train: {'INCOMPLETE' if TRAIN_INCOMPLETE else 'COMPLETE'} "
        f"({len(_done)} done, {len(_missing)} missing, {len(_failed)} failed to fit, {len(_infeasible)} infeasible)"
    )
else:
    print(f"train: skipped (MLC_STAGE={STAGE})")

RUN_ASSEMBLE = STAGE in ("assemble", "all") and not TRAIN_INCOMPLETE
if not RUN_ASSEMBLE:
    print("Remaining cells are skipped (RUN_ASSEMBLE is False): "
          + ("training is INCOMPLETE; resume with the same settings." if TRAIN_INCOMPLETE
             else f"MLC_STAGE={STAGE} does not assemble."))

Modules:   0%|          | 0/5 [00:00<?, ?module/s]

Embedders:   0%|          | 0/7 [00:00<?, ?embedder/s]

Methods & Algorithms:   0%|          | 0/32 [00:00<?, ?pair/s]

/home/cela/code/GitHub/msc-thesis/.conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/cela/code/GitHub/msc-thesis/.conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/m

HOMERClassifier(base_estimator=LogisticRegression(max_iter=10000,
                                                  random_state=42,
                                                  solver='liblinear')) failed to fit due to This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(1)
dict_values([{'name': 'MBR[RF] - GloVe.6B.50D', 'performance': 0.6824644549763034, 'model': MBRClassifier(base_estimator=RandomForestClassifier(random_state=42))}, {'name': 'HOMER[RF] - GloVe.6B.50D', 'performance': 0.6186046511627907, 'model': HOMERClassifier(base_estimator=RandomForestClassifier(random_state=42))}, {'name': 'LP[RF] - GloVe.6B.50D', 'performance': 0.5846153846153846, 'model': LPClassifier(base_estimator=RandomForestClassifier(random_state=42))}, {'name': 'LP[RF] - GloVe.6B.50D', 'performance': 0.7666666666666667, 'model': LPClassifier(base_estimator=RandomForestClassifier(random_state=42))}, {'name': 'RAKEL[RF] - GloVe.6B.50D', 'performan

Methods & Algorithms:   0%|          | 0/32 [00:00<?, ?pair/s]

... (68 outputs left out)

Methods & Algorithms:   0%|          | 0/32 [00:00<?, ?pair/s]

HOMERClassifier(base_estimator=LogisticRegression(max_iter=10000,
                                                  random_state=42,
                                                  solver='liblinear')) failed to fit due to This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(1)
dict_values([{'name': 'MBR[RF] - SBERT', 'performance': 0.8447488584474886, 'model': MBRClassifier(base_estimator=RandomForestClassifier(random_state=42))}, {'name': 'MBR[GNB] - SBERT', 'performance': 0.625, 'model': MBRClassifier(base_estimator=GaussianNB())}, {'name': 'MBR[LR] - SBERT', 'performance': 0.7, 'model': MBRClassifier(base_estimator=LogisticRegression(max_iter=10000, random_state=42,
                                                solver='liblinear'))}, {'name': 'MBR[GNB] - SBERT', 'performance': 0.6699029126213593, 'model': MBRClassifier(base_estimator=GaussianNB())}])
Hardware Performance
Accuracy:	71.79%
Hamming Loss:	0.112
                

Methods & Algorithms:   0%|          | 0/32 [00:00<?, ?pair/s]

HOMERClassifier(base_estimator=LogisticRegression(max_iter=10000,
                                                  random_state=42,
                                                  solver='liblinear')) failed to fit due to This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(1)
dict_values([{'name': 'BR[RF] - ATT&CK-BERT', 'performance': 0.8292682926829268, 'model': BRClassifier(base_estimator=RandomForestClassifier(random_state=42))}, {'name': 'BR[GNB] - ATT&CK-BERT', 'performance': 0.5957446808510638, 'model': BRClassifier(base_estimator=GaussianNB())}, {'name': 'MBR[LR] - ATT&CK-BERT', 'performance': 0.6829268292682927, 'model': MBRClassifier(base_estimator=LogisticRegression(max_iter=10000, random_state=42,
                                                solver='liblinear'))}, {'name': 'MBR[GNB] - ATT&CK-BERT', 'performance': 0.6810810810810811, 'model': MBRClassifier(base_estimator=GaussianNB())}])


## Stage: assemble (selection on validation)

Fails closed unless every expected embedding and candidate checkpoint (all modules x embedders x candidates) is present and verified. Per module and embedder: per-label best candidate on validation F1 (`extract_models`), then the embedder with the highest validation macro F1 (ties: the first embedder in `EMBEDDING_SOURCES`). Per-label selection and embedder choice use the same validation split, so the reported validation macro F1 is optimistic (equally so for every embedder). Candidates with a persisted fit-failure marker make assembly fail, unless `ALLOW_FAILED_CANDIDATES=1` (they are then excluded and listed in the output). Candidates recorded as infeasible by design (see above) are excluded from selection and listed in the output and summary without blocking. Assembly also refuses embedder inputs recorded as `unresolved` unless `ALLOW_UNRESOLVED_EMBEDDERS=1`. The non-production configuration and the embedder inputs are stored in the `metadata` attribute of the saved modules and System and in `mappings/*.pickle.meta.json`. The test set is used only in the evaluation below.

In [ ]:
# ASSEMBLY-CELL (marker: the provenance digest hashes the text of this cell)
ASSEMBLY_CELL_SOURCE = globals()["_ih"][-1]  # IPython input history: this cell
_FIRST_LINE = "# ASSEMBLY-CELL (marker: the provenance digest hashes the text of this cell)"
if ASSEMBLY_CELL_SOURCE.split("\n", 1)[0] != _FIRST_LINE or "def class_code_sha256(" not in ASSEMBLY_CELL_SOURCE:
    raise RuntimeError("assembly provenance: cannot obtain the source of the assembly cell (failing closed)")


def class_code_sha256(cls) -> dict:
    """sha256 of the source of every method/property/attribute of a cell-defined class
    (inspect.getsource per function, as for the run identity)."""
    out = {}
    for name, member in sorted(vars(cls).items()):
        if name in ("__module__", "__doc__", "__dict__", "__weakref__", "__qualname__", "__firstlineno__", "__static_attributes__"):
            continue
        funcs = [getattr(member, a) for a in ("fget", "fset", "__func__") if getattr(member, a, None) is not None]
        member = funcs[0] if funcs else member
        if inspect.isfunction(member):
            out[name] = hashlib.sha256(inspect.getsource(member).encode("utf-8")).hexdigest()
        else:  # generated descriptors (sklearn set_*_request) and class constants: type only, no address-bearing repr
            out[name] = type(member).__name__
    if not out:
        raise RuntimeError(f"assembly provenance: no source found for {cls.__name__}")
    return out


if RUN_ASSEMBLE:
    ck.verify_complete(store, EXPECTED_KEYS)
    for d in (MODELS_DIR, MAPPINGS_DIR):
        os.makedirs(d, exist_ok=True)
    # Embedder inputs stored with the embeddings: must be resolved (unless knowingly accepted).
    EMBEDDER_INPUTS = {e["name"]: store.load(fingerprint_key(e["name"])) for e in EMBEDDING_SOURCES}
    for _name, _stored in EMBEDDER_INPUTS.items():  # the embedder input must still be the one embedded
        check_embedder_input(_name, _stored, embedder_fingerprint(SOURCE_BY_NAME[_name], online=False), require=True)
    _unresolved = sorted(n for n, fp in EMBEDDER_INPUTS.items() if "unresolved" in fp.values())
    if _unresolved:
        print(f"WARNING: embedder input unresolved for {_unresolved}")
        if not allow_unresolved_embedders():
            raise RuntimeError(
                f"Embedder input unresolved for {_unresolved}; assembly refused "
                "(set ALLOW_UNRESOLVED_EMBEDDERS=1 to accept knowingly)."
            )
    # Persisted markers of the expected candidates (models are not kept: one record at a time).
    # Failures block (or are excluded knowingly); infeasible candidates (by design) never block.
    EXCLUDED, INFEASIBLE = {}, {}
    for level, embedder_name in ALL_UNITS:
        for candidate in CANDIDATES:
            key = candidate_key(level, embedder_name, candidate["name"])
            _record = store.load(key)
            if _record["embedder_input"] != EMBEDDER_INPUTS[embedder_name]:
                raise RuntimeError(
                    f"{key}: fitted on embedder input {_record['embedder_input']}, but the embeddings "
                    f"in the store come from {EMBEDDER_INPUTS[embedder_name]}"
                )
            if _record["error"] is not None:
                EXCLUDED[key] = _record["error"]
            elif _record["infeasible"] is not None:
                INFEASIBLE[key] = _record["infeasible"]
    # Assembly provenance: binds the assembly code, every input checkpoint (candidates, embeddings,
    # fingerprints: the selection is a function of the candidate records, so the selected ones are
    # covered), the validation split, the evaluation data, the excluded candidates and the run identity.
    PROVENANCE = {
        "run_identity": store.digest,
        "assembly_code": {
            "assembly_cell_sha256": hashlib.sha256(ASSEMBLY_CELL_SOURCE.encode("utf-8")).hexdigest(),
            "ModuleSystem": class_code_sha256(ModuleSystem),
            "System": class_code_sha256(System),
        },
        "checkpoint_sha256": {
            " / ".join(k): store.verify(k)["sha256"] for k in sorted(EXPECTED_KEYS)
        },
        "validation_split_sha256": {
            lvl: {f: EFFECTIVE_DATA[lvl][f] for f in ("n_rows", "labels_sha256", "classes", "train_idx_sha256", "val_idx_sha256")}
            for lvl in (m["level"] for m in SYSTEM_MODULES)
        },
        "test": {
            "texts_sha256": SPLIT_DIGEST["test"],
            "targets_sha256": text_digest([list(t) for t in targets]),
            "relevant_sha256": text_digest([bool(r) for r in test_tweets["relevant"]]),
        },
        "infeasible": sorted(" / ".join(k[1:]) for k in INFEASIBLE),
        "excluded_failed": sorted(" / ".join(k[1:]) for k in EXCLUDED),
    }
    PROVENANCE["digest"] = hashlib.sha256(
        json.dumps(PROVENANCE, sort_keys=True, ensure_ascii=True).encode("utf-8")
    ).hexdigest()
    print(f"RESULT/PROVENANCE: {PROVENANCE['digest']}")
    RUN_METADATA = {
        "provenance": PROVENANCE,
        "effective_data_sha256": EFFECTIVE_DATA["sha256"],
        "non_production": NON_PRODUCTION,
        "embedder_inputs": EMBEDDER_INPUTS,
        "train_texts_sha256": SPLIT_DIGEST["train"],
        "test_texts_sha256": SPLIT_DIGEST["test"],  # final evaluation texts (not part of the identity)
        "infeasible": {" / ".join(k[1:]): why for k, why in INFEASIBLE.items()},
        "excluded_failed": {" / ".join(k[1:]): why for k, why in EXCLUDED.items()},
    }
    ck.write_status(
        store.out_dir, "FAILED" if EXCLUDED and not ALLOW_FAILED_CANDIDATES else "COMPLETE",
        done=EXPECTED_KEYS, failed=EXCLUDED, identity_digest=store.digest,
    )
    if EXCLUDED:
        print(f"!!! {len(EXCLUDED)} candidate(s) failed to fit:")
        for _k, _msg in EXCLUDED.items():
            print(f"!!!   {_k[1]} / {_k[2]} / {_k[3]}: {_msg}")
        if not ALLOW_FAILED_CANDIDATES:
            raise RuntimeError(
                f"{len(EXCLUDED)} candidate(s) failed to fit (listed above); assembly refused. "
                "Fix the cause, or set ALLOW_FAILED_CANDIDATES=1 to exclude them knowingly."
            )
        print("!!! ALLOW_FAILED_CANDIDATES=1: the candidates above are EXCLUDED from selection.")
    if INFEASIBLE:
        print(f"{len(INFEASIBLE)} candidate(s) infeasible by design (documented data precondition; not an error, excluded from selection):")
        for _k, _why in INFEASIBLE.items():
            print(f"  {_k[1]} / {_k[2]} / {_k[3]}: {_why}")
    SELECTION = {}
    for module_config in tqdm(SYSTEM_MODULES, desc="Modules", unit="module"):
        best = None  # (macro F1, module, embedder name)

        for embedder in EMBEDDING_SOURCES:
            mlb, X_train, X_val, y_train, y_val = unit_data(
                module_config, store.load(embedding_key(embedder["name"], "train"))
            )
            models, failed = {}, []
            for candidate in CANDIDATES:
                record = store.load(
                    candidate_key(module_config["level"], embedder["name"], candidate["name"])
                )
                # the stored marker must agree with the labels it was decided on
                if infeasible_reason(candidate, y_train) != record["infeasible"]:
                    raise RuntimeError(
                        f"{module_config['level']} / {embedder['name']} / {candidate['name']}: stored feasibility "
                        f"{record['infeasible']!r} disagrees with the training labels ({infeasible_reason(candidate, y_train)!r})"
                    )
                if record["error"] is not None:
                    failed.append(candidate["name"])
                    continue
                if record["infeasible"] is not None:
                    continue
                models[f"{candidate['name']} - {embedder['name']}"] = {
                    "model": record["model"],
                    "assess": record["assess"],
                }
            if failed:
                print(f"{module_config['level']} / {embedder['name']}: {len(failed)} candidates failed to fit (skipped): {failed}")
            if not models:
                raise RuntimeError(f"{module_config['level']} / {embedder['name']}: no candidate was fitted")

            top_performing_models = su.extract_models(mlb.classes_, models, "f1-score")
            print({k: v["name"] for k, v in top_performing_models.items()})
            module = ModuleSystem(top_performing_models, mlb, metadata=RUN_METADATA)
            validation_results = su.assess(
                y_val, module.predict(X_val), module.predict_proba(X_val)
            )

            f1 = validation_results["report"]["macro avg"]["f1-score"]
            if best is None or f1 > best[0]:
                best = (f1, module, embedder["name"])
                print("==========================================================")
                print(f"{module_config['level'].capitalize()} Performance ({embedder['name']})")
                print("==========================================================")
                print(f"Accuracy:\t{validation_results['accuracy'] * 100:.2f}%")
                print(f"Hamming Loss:\t{validation_results['hamming_loss']:.3f}")
                report = pd.DataFrame(validation_results["report"])
                report.columns = list(module.classes) + [
                    "micro avg",
                    "macro avg",
                    "weighted avg",
                    "samples avg",
                ]
                report = (
                    report.transpose()
                    .map(lambda x: f"{x:.2f}" if isinstance(x, float) else x)
                    .to_string()
                )
                print(f"{report}")
                print("==========================================================\n")

            model_dictionary_filename = f"{MAPPINGS_DIR}/{module_config['level'].capitalize()}.{embedder['name']}"
            with open(f"{model_dictionary_filename}.pickle", "wb") as f:
                pickle.dump(models, f)
            with open(f"{model_dictionary_filename}.pickle.meta.json", "w") as f:  # provenance (the pickle format is unchanged)
                json.dump(RUN_METADATA, f, indent=1, sort_keys=True)

        SELECTION[module_config["level"]] = {
            "embedder": best[2],
            "validation macro f1": best[0],
            "models": {k: v["name"] for k, v in best[1].models.items()},
            "infeasible": [f"{k[2]} / {k[3]}: {why}" for k, why in INFEASIBLE.items() if k[1] == module_config["level"]],
        }
        module_filename = f"{MODELS_DIR}/{''.join(module_config['level'].capitalize().split(' '))}"
        with open(f"{module_filename}.pickle", "wb") as f:
            pickle.dump(best[1], f)

    print(json.dumps(SELECTION, indent=1))
    print(f"Embedder inputs: {json.dumps(EMBEDDER_INPUTS, sort_keys=True)}")
    if INFEASIBLE:
        print(f"{len(INFEASIBLE)} infeasible-by-design candidates (listed above, per module in the summary).")
    if NON_PRODUCTION:
        print(f"*** NON-PRODUCTION CONFIG: {NON_PRODUCTION} ***")
    if EXCLUDED:
        print(f"!!! {len(EXCLUDED)} EXCLUDED FAILED CANDIDATES (ALLOW_FAILED_CANDIDATES=1): "
              f"{[' / '.join(k[1:]) for k in EXCLUDED]}")

# 7. Model Evaluation

Held-out test set, assemble stage only (skipped when `RUN_ASSEMBLE` is False). The test texts are embedded once in the embed stage and passed to the System as cached embeddings. Label metrics use hard predictions; AUC uses the hierarchical scores of `System.predict_proba` (relevance AUC: the relevance score), which need not agree with the hard predictions when thresholded (see the classes section).

In [29]:
if RUN_ASSEMBLE:
    with open(f"{MODELS_DIR}/General.pickle", "rb") as f:
        gn_mod = pickle.load(f)

    with open(f"{MODELS_DIR}/Computersecurity.pickle", "rb") as f:
        cs_mod = pickle.load(f)

    with open(f"{MODELS_DIR}/Operatingsystems.pickle", "rb") as f:
        os_mod = pickle.load(f)

    with open(f"{MODELS_DIR}/Software.pickle", "rb") as f:
        sw_mod = pickle.load(f)

    with open(f"{MODELS_DIR}/Hardware.pickle", "rb") as f:
        hw_mod = pickle.load(f)

    TEST_EMBEDDINGS = {
        e["name"]: store.load(embedding_key(e["name"], "test")) for e in EMBEDDING_SOURCES
    }

    def test_embeddings(rows: slice) -> dict:
        return {name: X[rows] for name, X in TEST_EMBEDDINGS.items()}

In [30]:
if RUN_ASSEMBLE:
    solution = System(gn_mod, cs_mod, os_mod, sw_mod, hw_mod, metadata=RUN_METADATA)

In [21]:
if RUN_ASSEMBLE:
    display(solution.predict(texts[0:10], test_embeddings(slice(0, 10))))

array([[1., 0., 0., 0., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [22]:
if RUN_ASSEMBLE:
    display(solution.predict_targets(texts[0:10], test_embeddings(slice(0, 10))))

0    [antivirus and malware, databases, software, c...
1                                              [other]
2           [antivirus and malware, computer security]
3                                              [other]
4                                [databases, software]
5                                              [other]
6                                              [other]
7                                              [other]
8                                              [other]
9                                              [other]
dtype: object

In [23]:
if RUN_ASSEMBLE:
    display(
        pd.DataFrame(
            [
                solution.predict_targets(texts[10:20], test_embeddings(slice(10, 20))),
                targets[10:20],
                texts[10:20],
                test_tweets["relevant"][10:20],
            ],
            index=["Predicted", "Actual", "Text", "Relevant"],
        ).transpose()
    )

,Predicted,Actual,Text,Relevant
10,[other],[other],HOTEL MANAGEMENT - Data leak from Huazhu Hotel...,False
11,[other],"[databases, software]","Yes the vulnerability does exist and yes, the ...",True
12,[other],"[network security, computer security]",Healthier mentally &amp; more comfortable in m...,False
13,[other],"[antivirus and malware, computer security]",[] Slush doesn't give a sh*t about DDoS on his...,True
14,[other],[other],DLVRIT: US charges North Korean man in Sony ha...,False
15,"[antivirus and malware, databases, software, c...","[antivirus and malware, databases, software, c...",Designed by Tristar Software Solutions India S...,True
16,[other],"[network security, computer security]",Don’t call it a friendship unless you’re actua...,False
17,[other],[other],\nWe will be starting an exciting new project ...,False
18,"[antivirus and malware, computer security]","[network security, computer security]",attack from cripples Bank of website\n\n …,True
19,[other],"[computer, network security, hardware, compute...",7 bizarre security analogies about preventing ...,True


In [27]:
if RUN_ASSEMBLE:
    system_performance = solution.evaluate(texts, targets, TEST_EMBEDDINGS)
    print("==========================================================")
    print("System Performance")
    print("==========================================================")
    if NON_PRODUCTION:
        print(f"*** NON-PRODUCTION CONFIG: {NON_PRODUCTION} ***")
    if EXCLUDED:
        print(f"!!! {len(EXCLUDED)} EXCLUDED FAILED CANDIDATES: {[' / '.join(k[1:]) for k in EXCLUDED]}")
    print(f"Accuracy:\t{system_performance['accuracy'] * 100:.2f}%")
    print(f"Hamming Loss:\t{system_performance['hamming_loss']:.3f}")
    report = pd.DataFrame(system_performance["report"])
    report.columns = solution.classes() + [
        "micro avg",
        "macro avg",
        "weighted avg",
        "samples avg",
    ]
    report = (
        report.transpose()
        .map(lambda x: f"{x:.2f}" if isinstance(x, float) else x)
        .to_string()
    )
    print(f"{report}")
    print("==========================================================\n")

System Performance
Accuracy:	59.65%
Hamming Loss:	0.080
                      precision recall f1-score  support   auc
computer security          0.93   0.41     0.57  1738.00  0.70
hardware                   0.98   0.37     0.54   461.00  0.68
networking                 0.95   0.31     0.47   240.00  0.66
operating systems          0.95   0.56     0.71   156.00  0.78
other                      0.54   0.98     0.70  1603.00  0.72
software                   0.93   0.54     0.68   950.00  0.76
antivirus and malware      0.92   0.48     0.63  1110.00  0.73
network security           0.91   0.24     0.37   817.00  0.61
linux                      0.98   0.62     0.76    77.00  0.81
mac os                     1.00   0.46     0.63    13.00  0.73
unix                       0.82   0.64     0.72    14.00  0.82
windows                    0.88   0.46     0.60    46.00  0.73
databases                  0.92   0.60     0.73   342.00  0.80
computer                   0.97   0.36     0.52   362.00  0.68

In [25]:
if RUN_ASSEMBLE:
    Y = pd.Series([1 if t else 0 for t in test_tweets["relevant"]])
    Y_pred = pd.Series([not t for t in solution.predict(texts, TEST_EMBEDDINGS)[:, 4:5]])
    # AUC from the continuous relevance score, not from the hard predictions.
    Y_relevance = solution.relevance_score(texts, TEST_EMBEDDINGS)
    auc_score_macro = roc_auc_score(Y, Y_relevance, average="macro")
    auc_score_weighted = roc_auc_score(Y, Y_relevance, average="weighted")
    report = classification_report(Y, Y_pred, output_dict=True, zero_division=0.0)
    report["macro avg"]["auc"] = auc_score_macro
    report["weighted avg"]["auc"] = auc_score_weighted
    relevance_performance = {"hamming_loss": hamming_loss(Y, Y_pred), "report": report}

    print("==========================================================")
    print("System Performance")
    print("==========================================================")
    print(f"Hamming Loss:\t{relevance_performance['hamming_loss']:.3f}")
    report = pd.DataFrame(relevance_performance["report"])
    report.columns = ["not relevant", "relevant"] + [
        "accuracy",
        "macro avg",
        "weighted avg",
    ]
    report = (
        report.transpose()
        .map(lambda x: f"{x:.2f}" if isinstance(x, float) else x)
        .to_string()
    )
    print(f"{report}")
    print("==========================================================\n")

System Performance
Hamming Loss:	0.179
             precision recall f1-score  support   auc
not relevant      0.80   0.94     0.86  2444.00   nan
relevant          0.88   0.64     0.74  1623.00   nan
accuracy          0.82   0.82     0.82     0.82  0.82
macro avg         0.84   0.79     0.80  4067.00  0.79
weighted avg      0.83   0.82     0.81  4067.00  0.79



In [26]:
if RUN_ASSEMBLE:
    display(
        pd.DataFrame(
            np.hstack(
                (
                    Y.values.reshape(-1, 1),
                    Y_pred.values.reshape(-1, 1),
                    test_tweets["text"].values.reshape(-1, 1),
                )
            ),
            columns=["relevant", "predicted", "text"],
        )
    )

,relevant,predicted,text
0,1,True,threatmeter: irisgraphic sql injection vulnera...
1,0,False,Download this how-to guide on preparing for an...
2,1,True,The US Justice Department is poised to charge ...
3,0,False,"3) Control and monitor compliance. Also, in..."
4,1,True,"Ransomware down, but not out, report reveals"
...,...,...,...
4062,0,False,– with full knowledge of my increased vulnerab...
4063,1,False,CVE-2018-16460\n\nA command Injection in ps pa...
4064,1,True,threatmeter: To Video Converter Professional...
4065,1,False,"New vulnerability on the NVD on September 10, ..."


In [31]:
if RUN_ASSEMBLE:
    with open(f"{MODELS_DIR}/System.pickle", "wb") as f:
        pickle.dump(solution, f)

    del solution

In [32]:
if RUN_ASSEMBLE:
    with open(f"{MODELS_DIR}/System.pickle", "rb") as f:
        system = pickle.load(f)
    display(system)

System(cs=ModuleSystem(mlb=MultiLabelBinarizer(),
                       models={'0': {'model': MBRClassifier(base_estimator=RandomForestClassifier(random_state=42)),
                                     'name': 'MBR[RF] - DistilRoBERTa',
                                     'performance': 0.8093797276853253},
                               '1': {'model': MBRClassifier(base_estimator=RandomForestClassifier(random_state=42)),
                                     'name': 'MBR[RF] - DistilRoBERTa',
                                     'performance': 0.651731160896130...
                                     'performance': 0.8666666666666667}}),
       sw=ModuleSystem(mlb=MultiLabelBinarizer(),
                       models={'0': {'model': MBRClassifier(base_estimator=RandomForestClassifier(random_state=42)),
                                     'name': 'MBR[RF] - GloVe.6B.200D',
                                     'performance': 0.8316831683168316},
                               '1': {'model': MBRClassifier(base_estimator=RandomForestClassifier(random_state=42)),
                                     'name': 'MBR[RF] - GloVe.6B.200D',
                                     'performance': 0.9321357285429142}}))